In [ ]:
!pip install --upgrade google-cloud-bigquery vertexai pandas streamlit pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 991.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.8/131.8 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: google-cloud-storage
    Found existing installation: google-cloud-storage 3.10.1
    Uninstalling google-cloud-storage-3.10.1:
      Successfully uninstalled google-cloud-storage-3.10.1
  Attempting uninstall: google-cloud-aiplatform
    Found existing installation: google-cloud-aiplatform 1.147.0
    Uninstalling google-cloud-aiplatform-1.147.0:


In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from google.cloud import bigquery
import vertexai
from vertexai.generative_models import GenerativeModel

# set your project details
PROJECT_ID = "your-project-id"  # Enter your own project id here
LOCATION = "us-central1"

# Initialize clients
bq_client = bigquery.Client(project=PROJECT_ID)
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Load model
model = GenerativeModel("gemini-2.5-pro")

In [ ]:
# fetch KPIs query

kpi_query = """
select
  round(avg(burnout_score),1) as avg_burnout_score,
  round(countif(attrition_risk = 'High') * 100.0 / count(*), 1) as pct_high_attrition_risk,
  round(avg(hours_with_ai_assistance_daily),1) as avg_daily_ai_hours,
  round(avg(productivity_score),1) as avg_productivity_score,
  round(avg(ai_replaces_my_tasks_pct),1) as tasks_replaced
from my_db.workforce;

"""

kpi_df = bq_client.query(kpi_query).to_dataframe()
kpi_df

,avg_burnout_score,pct_high_attrition_risk,avg_daily_ai_hours,avg_productivity_score,tasks_replaced
0,50.1,5.7,4.2,57.5,41.2


In [ ]:
# task replaced by AI vs role

task_replacement_query = """
select job_role, round(avg(ai_replaces_my_tasks_pct),1) as task_replacement_percent
from my_db.workforce
group by job_role
order by  task_replacement_percent desc;
"""

task_replacement_df = bq_client.query(task_replacement_query).to_dataframe()
task_replacement_df

,job_role,task_replacement_percent
0,Backend Engineer,43.5
1,Software Engineer,43.4
2,Data Analyst,42.6
3,Cloud Architect,42.6
4,Data Scientist,42.5
5,AI Ethics Officer,42.0
6,DevOps Engineer,42.0
7,ML Engineer,41.9
8,Frontend Engineer,41.6
9,Product Manager,38.5


In [ ]:
# attrition risk by ai adoption

attrition_by_adoption_query = """
select ai_adoption_stage, attrition_risk,
count(*) as emp_count,
round(count(*) * 100.0/ sum(count(*)) over(partition by ai_adoption_stage),2) as percent
from my_db.workforce
group by ai_adoption_stage, attrition_risk
order by ai_adoption_stage, attrition_risk;
"""

attrition_by_adoption_df = bq_client.query(attrition_by_adoption_query).to_dataframe()
attrition_by_adoption_df

,ai_adoption_stage,attrition_risk,emp_count,percent
0,AI-First,High,16,6.40
1,AI-First,Low,120,48.00
2,AI-First,Medium,114,45.60
3,Experimenting,High,20,5.59
4,Experimenting,Low,184,51.40
5,Experimenting,Medium,154,43.02
6,Integrating,High,25,5.08
7,Integrating,Low,232,47.15
8,Integrating,Medium,235,47.76
9,Optimizing,High,24,6.00


In [ ]:
# fear of ai replacement

fear_of_ai_query = """
select fear_of_ai_replacement, round(count(*) * 100.0/ sum(count(*)) over (), 2) as percent
from my_db.workforce
group by fear_of_ai_replacement;

"""

fear_of_ai_df = bq_client.query(fear_of_ai_query).to_dataframe()
fear_of_ai_df

,fear_of_ai_replacement,percent
0,High,23.87
1,Low,34.87
2,Medium,41.27


In [ ]:
# high attrition risk cohort

high_attrition_query = """
select industry, round(avg(years_experience),2) as avg_YoE,
round(avg(burnout_score),2) as avg_burnout,
round(avg(job_satisfaction_1_5),2) as avg_job_satisfaction
from my_db.workforce
where fear_of_ai_replacement = 'High' and attrition_risk = 'High'
group by industry
order by avg_YoE desc;

"""

high_attrition_df = bq_client.query(high_attrition_query).to_dataframe()
high_attrition_df

,industry,avg_YoE,avg_burnout,avg_job_satisfaction
0,Consulting,14.40,63.00,2.72
1,Fintech,12.40,56.00,2.88
2,Media,12.22,60.00,2.88
3,E-commerce,9.85,60.46,2.65
4,Automotive,9.14,65.00,2.47
5,SaaS,9.00,62.67,2.65
6,Gaming,8.88,60.75,2.60
7,Healthtech,8.60,57.30,2.74
8,Cybersecurity,8.33,60.78,2.81
9,EdTech,6.43,54.29,2.83


In [ ]:
#Convert data to text format
kpi_context = kpi_df.to_string(index = False)
task_replacement_context = task_replacement_df.to_string(index = False)
attrition_by_adoption_context = attrition_by_adoption_df.to_string(index = False)
fear_of_ai_context = fear_of_ai_df.to_string(index = False)
high_attrition_context = high_attrition_df.to_string(index = False)

In [ ]:
# generate_insights function
def generate_insights(user_question):
    prompt = f"""
    You are a senior data analyst.

    Dataset Context:
    - 1500 tech employees (2026)
    - Focus: AI adoption vs burnout, productivity, and attrition

    Key KPIs:
    {kpi_context}

    Industry Attrition Insights:
    {high_attrition_context}

    Attrition Risk vs AI Adoption Stage Insights:
    {attrition_by_adoption_context}

    Percent of tasks replaced by AI for each job role Insights:
    {task_replacement_context}

    Fear of AI replacement Insights:
    {fear_of_ai_context}


    User Question:
    {user_question}
    Give insights, risks, and recommendations based on the user question.

    Instructions:
    1. Answer in 3 - 4 sentences only.
    2. Identify key patterns and trends
    3. Explain business impact
    4. Highlight risks (burnout, attrition, overuse of AI)
    4. Provide clear, actionable recommendations
    5. Keep response structured and concise with headings
    6. Only use the provided data. Do not assume missing values
    """
    response = model.generate_content(prompt)
    return response.text

In [ ]:
user_question = input("Ask a question about the dashboard")

# call the function
response = generate_insights(user_question)

Ask a question about the dashboardHow can we prevent attrition in the automotive industry?


In [ ]:
print(response)

### **Analysis of Attrition in the Automotive Industry**

**Key Insight**
The data reveals the automotive industry is a high-risk sector, exhibiting the highest average burnout score (65.00) and the lowest average job satisfaction (2.47) compared to all other industries.

**Risks & Business Impact**
This toxic combination of severe burnout and poor satisfaction is a primary driver for high employee attrition, threatening the retention of experienced staff (avg. 9.14 YoE) and leading to significant knowledge loss and increased hiring costs.

**Recommendation**
To prevent attrition, immediately prioritize targeted initiatives to diagnose and address the root causes of burnout while implementing programs to improve job satisfaction, as these are the most critical factors identified in the data for this industry.


In [ ]:
## Streamlit app

%%writefile app.py
import streamlit as st
import base64
from google.cloud import bigquery
import vertexai
from vertexai.generative_models import GenerativeModel

# -----------------------------
# INIT
# -----------------------------
# Set your project details
PROJECT_ID = "focus-arcanum-494414-i5"
LOCATION = "us-central1"

# Initialize clients
bq_client = bigquery.Client(project=PROJECT_ID)
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Load model
model = GenerativeModel("gemini-2.5-pro")

#-------- Load Data Context -------------

# fetch KPIs query
kpi_query = """
select
  round(avg(burnout_score),1) as avg_burnout_score,
  round(countif(attrition_risk = 'High') * 100.0 / count(*), 1) as pct_high_attrition_risk,
  round(avg(hours_with_ai_assistance_daily),1) as avg_daily_ai_hours,
  round(avg(productivity_score),1) as avg_productivity_score,
  round(avg(ai_replaces_my_tasks_pct),1) as tasks_replaced
from my_db.workforce;
"""

kpi_df = bq_client.query(kpi_query).to_dataframe()

#high attrition risk cohort
high_attrition_query = """
--High Attrition Risk Cohort
select industry, round(avg(years_experience),2) as avg_YoE,
round(avg(burnout_score),2) as avg_burnout,
round(avg(job_satisfaction_1_5),2) as avg_job_satisfaction
from my_db.workforce
where fear_of_ai_replacement = 'High' and attrition_risk = 'High'
group by industry
order by avg_YoE desc;

"""

high_attrition_df = bq_client.query(high_attrition_query).to_dataframe()


#attrition risk by ai adoption
attrition_by_adoption_query= """
--Attrition Risk By AI Adoption Stage
select ai_adoption_stage, attrition_risk,
count(*) as emp_count,
round(count(*) * 100.0/ sum(count(*)) over(partition by ai_adoption_stage),2) as percent
from my_db.workforce
group by ai_adoption_stage, attrition_risk
order by ai_adoption_stage, attrition_risk;

"""

attrition_by_adoption_df = bq_client.query(attrition_by_adoption_query).to_dataframe()

#tasks replaced by AI vs role
task_replacement_query = """
--task replacement % by role
select job_role, round(avg(ai_replaces_my_tasks_pct),1) as task_replacement_percent
from my_db.workforce
group by job_role
order by  task_replacement_percent desc;
"""

task_replacement_df = bq_client.query(task_replacement_query).to_dataframe()

#fear of ai replacement
fear_of_ai_query = """
select fear_of_ai_replacement, round(count(*) * 100.0/ sum(count(*)) over (), 2) as percent
from my_db.workforce
group by fear_of_ai_replacement;
"""

fear_of_ai_df = bq_client.query(fear_of_ai_query).to_dataframe()


#Convert Data to Text Context
kpi_context = kpi_df.to_string(index=False)
high_attrition_context = high_attrition_df.to_string(index=False)
attrition_by_adoption_context = attrition_by_adoption_df.to_string(index=False)
task_replacement_context =  task_replacement_df.to_string(index=False)
fear_of_ai_context = fear_of_ai_df.to_string(index=False)


# generate_insights function
def generate_insights(user_question):
    prompt = f"""
    You are a senior data analyst.

    Dataset Context:
    - 1500 employees (2026)
    - Focus: AI adoption vs burnout, productivity, and attrition

    Key KPIs:
    {kpi_context}

    Industry Attrition Insights:
    {high_attrition_context}

    Attrition Risk vs AI Adoption Stage Insights:
    {attrition_by_adoption_context}

    Percent of tasks replaced by AI for each job role Insights:
    {task_replacement_context}

    Fear of AI replacement Insights:
    {fear_of_ai_context}


    User Question:
    {user_question}

    Instructions to answer the user question:
    1. Answer straightforward in 3 - 4 sentences only
    2. Explain business impact
    3. Provide clear, actionable recommendations
    4. Keep response structured and concise with headings
    5. Only use the provided data. Do not assume missing values
    """
    response = model.generate_content(prompt)
    return response.text

#--- UI -----
st.set_page_config(layout="wide")

st.title("AI Workforce Dashboard + GenAI Insights")

# ---- LOAD DASHBOARD ----
with open("AI Workplace Attrition.pdf", "rb") as f:
    pdf_bytes = f.read()

base64_pdf = base64.b64encode(pdf_bytes).decode('utf-8')

pdf_display = f"""
<iframe src="data:application/pdf;base64,{base64_pdf}"
width="100%" height="600" type="application/pdf"></iframe>
"""

st.markdown("## Dashboard")
st.markdown(pdf_display, unsafe_allow_html=True)


# ---- QUESTION INPUT ----
st.markdown("## 🤖 Ask a question about the dashboard")

user_question = st.text_input("Enter your question:")

if st.button("Get Insights"):
    if user_question:
        st.write("### Response:")
        #from ai_workforce_attrition import generate_insights

        response = generate_insights(user_question)
        st.write(response)

Writing app.py


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("your-auth-token")  # Enter your own auth token here

In [ ]:
!pkill -f streamlit
!streamlit run app.py &>/dev/null &

In [ ]:
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://wannabe-geek-giddiness.ngrok-free.dev" -> "http://localhost:8501"
